## QDM correction to CESM2-WACCM so
This code applies a Quantile Delta Mapping (QDM) bias correction to the gridded ocean salinity in CESM2-WACCM output.  The basis of the QDM correction is the EN4 (Hadley) reanalysis.  We take the 30-year period from 1985-2014 as the climatological baseline for this correction.

Note that the GCM salinity and the EN4 salinity must be on consistent grids before applying QDM.  In the current workflow, both are regridded onto a common horizontal grid in Step2.  If you are processing a new GCM, you may need to tweak the regridding routine.

Note that **you must do the QDM correction on the full time period, 1850-2100 (or 2300), all at once.**  It is correcting the cumulative distribution function, which will differ between sliced time periods versus the full time period.  This example script corrects only the historical period; you should instead process all time slices through Step1 and Step2, then apply this script to a multifile dataset with all slices included.  The `xr.open_mfdataset` command in the CMIP model read-in cell facilitates this.


19 Mar 2026 | EHU

### Setup

In [ ]:
SelModel = 'CESM2-WACCM'
BaselinePeriod = ['1985','2014'] ## climatological baseline for QDM
YearsIncluded = ['1850', '2300'] ## set these to the endpoints if you want to slice the CESM2 data.
## EN4 data run 1950-2014, but we will use it only during the BaselinePeriod anyway

DirSave = f'/Users/eultee/Desktop/'
DirIn = f'/Users/eultee/Desktop/'

DirHadley = f'/Users/eultee/Desktop/'
HadleyFile = f'/so-Hadley-1950_2020_regrid.nc'

### Imports

In [ ]:
import os
import sys
import copy
import csv
import time
import datetime
import cftime
import math
import cartopy.crs as ccrs ## map projections
import pandas as pd
import numpy as np
import xarray as xr
import netCDF4 as nc
import matplotlib.pyplot as plt

from pyproj import CRS

In [ ]:
## from cmethods.utils
import warnings
from typing import TYPE_CHECKING, Optional, Union, TypeVar

XRData_t = (xr.Dataset, xr.DataArray)
NPData_t = (list, np.ndarray, np.generic)
XRData = TypeVar("XRData", xr.Dataset, xr.DataArray)
NPData = TypeVar("NPData", list, np.ndarray, np.generic)
MAX_SCALING_FACTOR = 2 ## to allow multiplicative correction?


def check_xr_types(obs: XRData, simh: XRData, simp: XRData) -> None:
    """
    Checks if the parameters are in the correct type. **only used internally**
    """
    phrase: str = "must be type xarray.core.dataarray.Dataset or xarray.core.dataarray.DataArray"

    if not isinstance(obs, XRData_t):
        raise TypeError(f"'obs' {phrase}")
    if not isinstance(simh, XRData_t):
        raise TypeError(f"'simh' {phrase}")
    if not isinstance(simp, XRData_t):
        raise TypeError(f"'simp' {phrase}")

def check_np_types(
    obs: NPData,
    simh: NPData,
    simp: NPData,
) -> None:
    """
    Checks if the parameters are in the correct type. **only used internally**
    """
    phrase: str = "must be type list, np.ndarray or np.generic"

    if not isinstance(obs, NPData_t):
        raise TypeError(f"'obs' {phrase}")
    if not isinstance(simh, NPData_t):
        raise TypeError(f"'simh' {phrase}")
    if not isinstance(simp, NPData_t):
        raise TypeError(f"'simp' {phrase}")

def nan_or_equal(value1: float, value2: float) -> bool:
    """
    Returns True if the values are equal or at least one is NaN

    :param value1: First value to check
    :type value1: float
    :param value2: Second value to check
    :type value2: float
    :return: If any value is NaN or values are equal
    :rtype: bool
    """
    return np.isnan(value1) or np.isnan(value2) or value1 == value2
        
def ensure_dividable(
    numerator: Union[float, np.ndarray],
    denominator: Union[float, np.ndarray],
    max_scaling_factor: float,
) -> np.ndarray:
    """
    Ensures that the arrays can be divided. The numerator will be multiplied by
    the maximum scaling factor of the CMethods class if division by zero.

    :param numerator: Numerator to use
    :type numerator: np.ndarray
    :param denominator: Denominator that can be zero
    :type denominator: np.ndarray
    :return: Zero-ensured division
    :rtype: np.ndarray | float
    """
    with np.errstate(divide="ignore", invalid="ignore"):
        result = numerator / denominator

    if isinstance(numerator, np.ndarray):
        mask_inf = np.isinf(result)
        result[mask_inf] = numerator[mask_inf] * max_scaling_factor  # type: ignore[index]

        mask_nan = np.isnan(result)
        result[mask_nan] = 0  # type: ignore[index]
    elif np.isinf(result):
        result = numerator * max_scaling_factor
    elif np.isnan(result):
        result = 0.0

    return result

def get_pdf(
    x: Union[list, np.ndarray],
    xbins: Union[list, np.ndarray],
) -> np.ndarray:
    r"""
    Compuites and returns the the probability density function :math:`P(x)`
    of ``x`` based on ``xbins``.

    :param x: The vector to get :math:`P(x)` from
    :type x: list | np.ndarray
    :param xbins: The boundaries/bins of :math:`P(x)`
    :type xbins: list | np.ndarray
    :return: The probability densitiy function of ``x``
    :rtype: np.ndarray

    .. code-block:: python
        :linenos:
        :caption: Compute the probability density function :math:`P(x)`

        >>> from cmethods get_pdf

        >>> x = [1, 2, 3, 4, 5, 5, 5, 6, 7, 8, 9, 10]
        >>> xbins = [0, 3, 6, 10]
        >>> print(get_pdf(x=x, xbins=xbins))
        [2, 5, 5]
    """
    pdf, _ = np.histogram(x, xbins)
    return pdf


def get_cdf(
    x: Union[list, np.ndarray],
    xbins: Union[list, np.ndarray],
) -> np.ndarray:
    r"""
    Computes and returns returns the cumulative distribution function :math:`F(x)`
    of ``x`` based on ``xbins``.

    :param x: Vector to get :math:`F(x)` from
    :type x: list | np.ndarray
    :param xbins: The boundaries/bins of :math:`F(x)`
    :type xbins: list | np.ndarray
    :return: The cumulative distribution function of ``x``
    :rtype: np.ndarray


    .. code-block:: python
        :linenos:
        :caption: Compute the cumulative distribution function :math:`F(x)`

        >>> from cmethods.utils import get_cdf

        >>> x = [1, 2, 3, 4, 5, 5, 5, 6, 7, 8, 9, 10]
        >>> xbins = [0, 3, 6, 10]
        >>> print(get_cdf(x=x, xbins=xbins))
        [0.0, 0.16666667, 0.58333333, 1.]
    """
    pdf, _ = np.histogram(x, xbins)
    cdf = np.insert(np.cumsum(pdf), 0, 0.0)
    return cdf / cdf[-1]


def get_inverse_of_cdf(
    base_cdf: Union[list, np.ndarray],
    insert_cdf: Union[list, np.ndarray],
    xbins: Union[list, np.ndarray],
) -> np.ndarray:
    r"""
    Returns the inverse cumulative distribution function as:
    :math:`F^{-1}_{x}\left[y\right]` where :math:`x` represents ``base_cdf`` and
    ``insert_cdf`` is represented by :math:`y`.

    :param base_cdf: The basis
    :type base_cdf: list | np.ndarray
    :param insert_cdf: The CDF that gets inserted
    :type insert_cdf: list | np.ndarray
    :param xbins: Probability boundaries
    :type xbins: list | np.ndarray
    :return: The inverse CDF
    :rtype: np.ndarray
    """
    return np.interp(insert_cdf, base_cdf, xbins)


In [ ]:
def quantile_delta_mapping(
    obs: NPData,
    simh: NPData,
    simp: NPData,
    n_quantiles: int,
    kind: str = "+",
    **kwargs,
    ) -> NPData:
    r"""
    Based on https://python-cmethods.readthedocs.io/en/latest/methods.html#quantile-delta-mapping

    kind: str, default + for additive, can be set to * for multiplicative
    """
    # check_adjust_called(
    #     function_name="quantile_delta_mapping",
    #     adjust_called=kwargs.get("adjust_called"),
    # )
    check_np_types(obs=obs, simh=simh, simp=simp)

    if not isinstance(n_quantiles, int):
        raise TypeError("'n_quantiles' must be type int")

    if kind=='+':
        obs, simh, simp = (
            np.array(obs),
            np.array(simh),
            np.array(simp),
        )  # to achieve higher accuracy
        global_max = kwargs.get("global_max", max(np.nanmax(obs), np.nanmax(simh)))
        global_min = kwargs.get("global_min", min(np.nanmin(obs), np.nanmin(simh)))

        if nan_or_equal(value1=global_max, value2=global_min):
            return simp

        wide = abs(global_max - global_min) / n_quantiles
        xbins = np.arange(global_min, global_max + wide, wide)

        cdf_obs = get_cdf(obs, xbins)
        cdf_simh = get_cdf(simh, xbins)
        cdf_simp = get_cdf(simp, xbins)

        # calculate exact CDF values of $F_{sim,p}[T_{sim,p}(t)]$
        epsilon = np.interp(simp, xbins, cdf_simp)  # Eq. 1.1
        QDM1 = get_inverse_of_cdf(cdf_obs, epsilon, xbins)  # Eq. 1.2
        delta = simp - get_inverse_of_cdf(cdf_simh, epsilon, xbins)  # Eq. 1.3
        return QDM1 + delta  # Eq. 1.4

    if kind=='*':
        obs, simh, simp = np.array(obs), np.array(simh), np.array(simp)
        global_max = kwargs.get("global_max", max(np.nanmax(obs), np.nanmax(simh)))
        global_min = kwargs.get("global_min", 0.0)
        if nan_or_equal(value1=global_max, value2=global_min):
            return simp

        wide = global_max / n_quantiles
        xbins = np.arange(global_min, global_max + wide, wide)

        cdf_obs = get_cdf(obs, xbins)
        cdf_simh = get_cdf(simh, xbins)
        cdf_simp = get_cdf(simp, xbins)

        epsilon = np.interp(simp, xbins, cdf_simp)  # Eq. 1.1
        QDM1 = get_inverse_of_cdf(cdf_obs, epsilon, xbins)  # Eq. 1.2

        delta = ensure_dividable(  # Eq. 2.3
            simp,
            get_inverse_of_cdf(cdf_simh, epsilon, xbins),
            max_scaling_factor=kwargs.get(
                "max_scaling_scaling",
                MAX_SCALING_FACTOR,
            ),
        )
        return QDM1 * delta  # Eq. 2.4
    raise NotImplementedError(
        f"{kind=} not available for quantile_delta_mapping. Use '+' or '*' instead.",
    )


def apply_cmfunc(
    method: str,
    obs: XRData,
    simh: XRData,
    simp: XRData,
    **kwargs: dict,
) -> XRData:
    """
    Internal function used to apply the bias correction technique to the
    passed input data.
    """
    ## hard-code the QDM method
    if method!='quantile_delta_mapping':
        raise UnknownMethodError('Not implemented for methods other than quantile_delta_mapping')
        ## give this a default for what we want to do
    else:
        method='quantile_delta_mapping' ## not actually going to use this
    
    check_xr_types(obs=obs, simh=simh, simp=simp)
    # if method not in __METHODS_FUNC__:
    #     raise UnknownMethodError(method, __METHODS_FUNC__.keys())

    if kwargs.get("input_core_dims"):
        if not isinstance(kwargs["input_core_dims"], dict):
            raise TypeError("input_core_dims must be an object of type 'dict'")
        if not len(kwargs["input_core_dims"]) == 3 or any(
            not isinstance(value, str) for value in kwargs["input_core_dims"].values()
        ):
            raise ValueError(
                'input_core_dims must have three key-value pairs like: {"obs": "time", "simh": "time", "simp": "time"}',
            )

        input_core_dims = kwargs.pop("input_core_dims")
    else:
        input_core_dims = {"obs": "time", "simh": "time", "simp": "time"}

    result: XRData = xr.apply_ufunc(
        quantile_delta_mapping,
        obs,
        simh,
        # Need to spoof a fake time axis since 'time' coord on full dataset is
        # different than 'time' coord on training dataset.
        simp.rename({input_core_dims["simp"]: "__t_simp__"}),
        dask="parallelized",
        vectorize=True,
        # This will vectorize over the time dimension, so will submit each grid
        # cell independently
        input_core_dims=[
            [input_core_dims["obs"]],
            [input_core_dims["simh"]],
            ["__t_simp__"],
        ],
        # Need to denote that the final output dataset will be labeled with the
        # spoofed time coordinate
        output_core_dims=[["__t_simp__"]],
        kwargs=dict(kwargs),
    )

    # Rename to proper coordinate name.
    result = result.rename({"__t_simp__": input_core_dims["simp"]})

    # ufunc will put the core dimension to the end (time), so want to preserve
    # original order where time is commonly first.
    return result.transpose(*obs.rename({input_core_dims["obs"]: input_core_dims["simp"]}).dims)


In [ ]:
## Time utils from Bryan Riel
## pasting stuff from iceutils below.
#-*- coding: utf-8 -*-

def tdec2datestr(tdec_in, returndate=False):
    """
    Convert a decimaly year to an iso date string.
    """
    if isinstance(tdec_in, (list, np.ndarray)):
        tdec_list = copy.deepcopy(tdec_in)
    else:
        tdec_list = [tdec_in]
    current_list = []
    for tdec in tdec_list:
        year = int(tdec)
        yearStart = datetime.datetime(year, 1, 1)
        if year % 4 == 0:
            ndays_in_year = 366.0
        else:
            ndays_in_year = 365.0
        days = (tdec - year) * ndays_in_year
        seconds = (days - int(days)) * 86400
        tdelta = datetime.timedelta(days=int(days), seconds=int(seconds))
        current = yearStart + tdelta
        if not returndate:
            current = current.isoformat(' ').split()[0]
        current_list.append(current)

    if len(current_list) == 1:
        return current_list[0]
    else:
        return np.array(current_list)


def datestr2tdec(yy=0, mm=0, dd=0, hour=0, minute=0, sec=0, microsec=0, dateobj=None):
    """
    Convert year, month, day, hours, minutes, seconds to decimal year.
    """
    if dateobj is not None:
        if type(dateobj) == str:
            yy, mm, dd = [int(val) for val in dateobj.split('-')]
            hour, minute, sec = [0, 0, 0]
        elif type(dateobj) == datetime.datetime:
            attrs = ['year', 'month', 'day', 'hour', 'minute', 'second']
            yy, mm, dd, hour, minute, sec = [getattr(dateobj, attr) for attr in attrs]
        elif type(dateobj) == np.datetime64:
            yy = dateobj.astype('datetime64[Y]').astype(int) + 1970
            mm = dateobj.astype('datetime64[M]').astype(int) % 12 + 1
            days = (
                (dateobj - dateobj.astype('datetime64[M]')) / np.timedelta64(1, 'D')
            )
            dd = int(days) + 1
            hour, minute, sec = [0, 0, 0]
        else:
            raise NotImplementedError('dateobj must be str, datetime, or np.datetime64.')

    # Make datetime object for start of year
    yearStart = datetime.datetime(yy, 1, 1, 0, 0, 0)
    # Make datetime object for input time
    current = datetime.datetime(yy, mm, dd, hour, minute, sec, microsec)
    # Compute number of days elapsed since start of year
    tdelta = current - yearStart
    # Convert to decimal year and account for leap year
    if yy % 4 == 0:
        return float(yy) + tdelta.total_seconds() / (366.0 * 86400)
    else:
        return float(yy) + tdelta.total_seconds() / (365.0 * 86400)

### Load in data

In [ ]:
## Load EN4 using xarray
ds1 = xr.open_dataset(DirHadley+HadleyFile, decode_times='timeDim')
ds1

In [ ]:
## load in CESM TF for all time slices available, using multifile dataset
## update the wildcard as needed for your directory structure
ds3 = xr.open_mfdataset(f'{DirIn}/ssp585/so_Omon_CESM2*regrid.nc').sel(time=slice(YearsIncluded[0], YearsIncluded[1]))
ds3

In [ ]:
### CHECK for salinity

# with dask.config.set(**{'array.slicing.split_large_chunks': True}): ## mitigate performance problem with slicing
ds_m = ds3.where(ds3.so<1e20)

ds_m.sel(time=slice('1850', '1900')).mean().compute()

In previous processing of ocean TF, depth resampling seemed to be smearing fill value across depth levels, so that the mean of depth-resampled dataset was wrong, even when fill values masked out.  Created `ds_m`  before resampling to address this issue.  Check mean below.

### Apply DateTimeIndex
Previously, we applied a DatetimeIndex to the CESM values so that it could be compared directly with EN4 values.  However, dates after year 2262 fall outside the valid range for datetime64.  Convert EN4 to cftime index instead.

In [ ]:
test_ds_full = ds_m.so.sel(time=slice(YearsIncluded[0], YearsIncluded[1]))

## make it a Dataset
test_ds_full = test_ds_full.to_dataset()
test_ds_full

In [ ]:
ds1_temp = ds1.rename({'Salinity': 'so'})
tobs_ds_full = ds1_temp.so

## convert the index to cftime
tobs_ds_full = tobs_ds_full.convert_calendar('noleap')
tobs_ds_full

## Apply QDM correction
We will apply the QDM correction on annual, de-trended series with the monthly residual variability re-applied afterward.

In [ ]:
## name the datasets to match what the rest of the code expects
tsim_match = test_ds_full
tobs_repr_match = tobs_ds_full

### Separate annual and monthly var

In [ ]:
annual_ds = tsim_match.resample(time='YE').mean()

residual_ds = tsim_match.resample(time='ME').ffill() - annual_ds.resample(time='ME').ffill()

### Detrend the data to be fit
QDM performs poorly when values in the "projection" series exceed those seen in the "historical".  Detrend the future, historical, and reanalysis series before QDM correction.

In [ ]:
def detrend_ser(da, dim, deg=1, var='sodpavg0to500_bathymin100', return_fit=True):
    ## based on Gist by Ryan Abernathey
    ## hard coding Dataset version for now to make it behave with our datasets
    
    p = da.polyfit(dim=dim, deg=deg)
    
    if type(da) is xr.core.dataarray.DataArray:
        fit=xr.polyval(da[dim], p.polyfit_coefficients)
    elif type(da) is xr.core.dataset.Dataset:
        fit = xr.polyval(da[dim], p.so_polyfit_coefficients) 
        ## eventually use `var` argument to take any variable of interest
        ## for now hard-coded the name of the depth-averaged thermal forcing, so this
        ## will fail if we change that name
    else:
        print("Unrecognized input type. Expected xarray DataArray or Dataset, got {}".format(type(da)))

    if return_fit==True: ## give back the fitted values to use in reconstructing a series
        return da-fit, fit
    else:
        return da-fit

detrended_obs = detrend_ser(tobs_repr_match.sel(time=slice(BaselinePeriod[0],BaselinePeriod[1])).resample(time='A').mean(),
                            dim='time',
                            deg=1)[0]

detrended_simh = detrend_ser(annual_ds.sel(time=slice(BaselinePeriod[0], BaselinePeriod[1])),
                             dim='time',
                             deg=1)[0]
# detrended_simp = detrend_ser(annual_ds.sel(time=slice(annual_ds.time[0], BaselinePeriod[0])),
#                              dim='time',
#                              deg=1)[0]

## correct the full time series
detrended_simp = detrend_ser(annual_ds,
                             dim='time',
                             deg=1)[0]


In [ ]:
## produce the QDM fit on detrended data
qdm_detrended = apply_cmfunc(
        method = "quantile_delta_mapping",
        obs = detrended_obs,
        simh = detrended_simh.rename({'time':'t_simh'}).chunk(dict(t_simh=-1)),
        simp = detrended_simp.chunk(dict(time=-1)),
        n_quantiles = 100,
        input_core_dims={"obs": "time", "simh": "t_simh", "simp": "time"},
        kind = "+", # to calculate the relative rather than the absolute change, "*" can be used instead of "+" (this is prefered when adjusting precipitation)
    )

In [ ]:
## reconstruct dataset from reanalysis mean val, future trend, QDM, and monthly residual
reanalysis_meanval = tobs_repr_match.sel(time=slice(BaselinePeriod[0], BaselinePeriod[1])).mean(dim='time')
CESM_meanval = annual_ds.sel(time=slice(BaselinePeriod[0], BaselinePeriod[1])).mean(dim='time')

## The same structure applies whether a future or past period
baseline_trendonly = detrend_ser(annual_ds,
                             dim='time',
                             deg=1)[1]

baseline_trend_series = reanalysis_meanval - CESM_meanval + baseline_trendonly ## remove model mean and replace it with reanalysis

qdm_dtr_series = qdm_detrended.so
qdm_plus_resid = baseline_trend_series.resample(time='ME').ffill() + qdm_dtr_series.resample(time='ME').ffill() + residual_ds.so


qdm_plus_resid

In [ ]:
qdm_plus_resid.mean().compute()

In [ ]:
test_ds_full.mean().compute()

### Test visualization (optional)
This generates a map plot at the 200-m depth level.  The salinity of the corrected dataset (leftmost map) should be positive and agree in order of magnitude with the means computed above.  The middle plot shows the difference between the QDM-corrected and the original CMIP model salinity, and the rightmost plot shows the difference between QDM-corrected and original EN4 reanalysis salinity.  Both difference plots should have a smaller order of magnitude than the corrected dataset, with both positive and negative values.  The differences between CMIP model and QDM (middle plot) will likely be larger than those between EN4 and QDM (right plot) as long as you plot a date during the calibration period (1985-2014).

In [ ]:
## what does it look like?
## this is commented by default, but you may wish to uncomment it to check your processing.

depth_toplot=200
date_toplot = '2010-01-30'
date_toplot_cftime = cftime.DatetimeNoLeap(*map(int, date_toplot.split('-'))) # Simple example for YYYY-MM-DD
so_toplot_sim = test_ds_full.so.sel(time=date_toplot_cftime, depth=depth_toplot, method='nearest')
so_toplot_obs = tobs_ds_full.sel(time=date_toplot_cftime, depth=depth_toplot, method='nearest')
so_toplot_qdm = qdm_plus_resid.so.sel(time=date_toplot_cftime, depth=depth_toplot, method='nearest')

# obs_x, obs_y = np.meshgrid(tobs_ds_full.lon, tobs_ds_full.lat)

### Limits of Greenland domain ###
limN           = 86.0 ## degrees N latitude
limS           = 57.0 ## degrees N latitude
limE           = 4.0 ## degrees E latitude
limW           = 274.0 ## degrees E latitude

GrIS_polar_stereo = ccrs.Stereographic(
            central_latitude=90.0,
            central_longitude=-45.0,
            false_easting=0.0,
            false_northing=0.0,
            true_scale_latitude=70.0,
            globe=ccrs.Globe('WGS84')
        )

fig, axs = plt.subplots(1,3, layout='constrained',
                        subplot_kw={'projection': GrIS_polar_stereo, 'extent':  [-65, -20, limS, limN]})

sc_qdm = axs[0].scatter(y=so_toplot_qdm.lat, x=so_toplot_qdm.lon, 
                        c=so_toplot_qdm, transform=ccrs.PlateCarree(),
                        cmap='viridis');
axs[0].set(aspect=1, title='CESM2-WACCM w/ QDM')
axs[0].coastlines()
cbar1 = plt.colorbar(sc_qdm, orientation='vertical', label='so', shrink=0.4)

sc_simdiff = axs[1].scatter(y=so_toplot_sim.lat, x=so_toplot_sim.lon, 
                        c=so_toplot_qdm-so_toplot_sim, transform=ccrs.PlateCarree(),
                        cmap='RdBu', vmin=-2., vmax=2.);
axs[1].set(aspect=1, title='diff QDM - model')
axs[1].coastlines()
# cbar2 = plt.colorbar(sc_obs, orientation='vertical', label='so [°C]', shrink=0.4)

sc_obsdiff = axs[2].scatter(y=so_toplot_qdm.lat, x=so_toplot_qdm.lon, 
                        c=so_toplot_qdm-so_toplot_obs, transform=ccrs.PlateCarree(),
                        cmap='RdBu', vmin=-2., vmax=2.);
axs[2].set(aspect=1, title='diff QDM - EN4')
axs[2].coastlines()
cbar2 = plt.colorbar(sc_obsdiff, orientation='vertical', label='so diff', shrink=0.4)

fig.suptitle('{} - {}m depth'.format(date_toplot, depth_toplot))

### Write out

In [ ]:
from datetime import datetime, date

now = datetime.now()
ds_temp = qdm_plus_resid ## this one is already a dataset
ds_out = ds_temp.assign_attrs(title='QDM-corrected ocean salinity for {}'.format(SelModel),
                             summary='Salinity extracted from {} in a bounding'.format(SelModel) + 
                              ' box around Greenland, for ISMIP7 Greenland forcing. ' +
                              ' Additive QDM correction applied to annual based on Hadley data, with' +
                              ' monthly residual added',
                             institution='NASA Goddard Space Flight Center',
                             creation_date=now.strftime('%Y-%m-%d %H:%M:%S'))

ds_out

In [ ]:
ds_reduced = ds_out.astype('float32') ## set encoding to reduce file size
ds_reduced

In [ ]:
ds_out.sel(time=slice(cftime.DatetimeNoLeap(1850, 1, 1), cftime.DatetimeNoLeap(2014, 12, 31)))

In [ ]:
# The first YearTags is the set of slices desired for ISMIP forcing files.
# Note that the process of writing the netcdf (done in the next code cell) can take a lot of memory. If 
# this is being done on a machine with limited memory for the full time spans, we've found that the 
# kernel can crash. To perform testing, you may choose to write smaller time spans (e.g., 10 years, as
# specified in the second YearTags below.
YearTags = [(1850,2014), 
            (2015,2100),
            (2101,2300)]

# YearTags = [(1850,2014),]

ds_splits = [ds_reduced.sel(time=slice(cftime.DatetimeNoLeap(yt[0], 1, 1), cftime.DatetimeNoLeap(yt[1], 12, 31))) for yt in YearTags]
ds_splits[0]

In [ ]:
from dask.diagnostics import ProgressBar

for yt, ds in zip(YearTags, ds_splits):
    print('Processing years {}-{}'.format(yt[0],yt[1]))
    out_fn = DirSave + '/soQDM_additive-AllLevels-CommonGrid-{}-{}_{}-{}.nc'.format(SelModel, 
                                                                yt[0], yt[1],
                                                                now.strftime('%Y%m%d'))
    
    
    
    with ProgressBar():
        ds.to_netcdf(path=out_fn)
